# YoloV8 Classification
Este Notebook es parte de un proyecto que se puede encontrar [aqui](https://github.com/nel-eleven11/Proyecto2_DataScience), donde se busca diseñar una aplicación de datos para comparar e interactuar con diferentes modelos de visión por computadora. Primero, vamos a empezar instalando las librerías requeridas que incluyen Ultralytics y los modelos Yolo. Adicionalmente, estaremos utilizando Polars en lugar de Pandas por conflictos de dependencias dentro de Kaggle.

In [1]:
# Minimal, let ultralytics bring its own friends
!pip install --upgrade --no-cache-dir "ultralytics[export]" "opencv-python-headless" "polars"

from ultralytics import YOLO
import os
import numpy as np
import polars as pl
from pathlib import Path
import shutil

INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 248.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 370.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 66.4 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 155.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 MB 58.7 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 87.7 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 210.7 MB/s eta 0:00:00
   ━━━━━

Según el output, Ultralytics se instaló de manera correcta. Sin embargo, tenemos un warning por parte de Pip sobre algunas dependencias que podemos ignorar. También podemos realizar un sanity-check simple para verificar que todo esté funcionando

In [2]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.info()

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs


(129, 3157200, 0, 8.8575488)

## Pre-Procesamiento
A pesar de ya haber realizado un EDA, todavía debemos de preparar los datos en un formato soportado por YoloV8. Primero, vamos a empezar cargando los datos de nuestro dataset.

### Carga de Datos
Al trabajar dentro de Kaggle, podemos importar los datos y los outputs del Notebook de limpieza. Podemos revisar los directorios rápidamente

In [3]:
import os
print(os.listdir("/kaggle/input"))

['00-eda-and-cleaning', 'mosquito-data']


Luego, podemos setear algunas variables que nos serán de utilidad para saber dónde se encuentra la información.

In [4]:
RAW_DATA_PATH = "/kaggle/input/mosquito-data"
EDA_OUTPUT_PATH = "/kaggle/input/00-eda-and-cleaning"

print("raw:", os.listdir(RAW_DATA_PATH))
print("eda:", os.listdir(EDA_OUTPUT_PATH))

raw: ['train_images', 'sample_submission_phase1 (1).csv', 'test_images_phase1', 'test_phase1.csv', 'train.csv']
eda: ['__results__.html', 'val.csv', '__notebook__.ipynb', '__results___files', '__output__.json', 'train.csv', 'test.csv', 'custom.css']


Podemos ver por los resultados, que tenemos cargados ya los resultados de la limpieza en EDA_OUTPUT_PATH/train.csv, test.csv y val.csv respectivamente. Adicionalmente, las imágenes que utilizaremos se encuentran en RAW_DATA_PATH/train_images. Podemos cargar los datos hacia DataFrames utilizando Polars.

In [5]:
train_df = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "train.csv"))
val_df   = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "val.csv"))
test_df  = pl.read_csv(os.path.join(EDA_OUTPUT_PATH, "test.csv"))

print("train shape:", train_df.shape)
print("val shape  :", val_df.shape)
print("test shape :", test_df.shape)

print("Columns:", train_df.columns)
print("Class labels:", train_df.select("class_label").unique())

train shape: (6396, 8)
val shape  : (800, 8)
test shape : (800, 8)
Columns: ['img_fName', 'img_w', 'img_h', 'bbx_xtl', 'bbx_ytl', 'bbx_xbr', 'bbx_ybr', 'class_label']
Class labels: shape: (6, 1)
┌────────────────────┐
│ class_label        │
│ ---                │
│ str                │
╞════════════════════╡
│ albopictus         │
│ culiseta           │
│ culex              │
│ aegypti            │
│ japonicus/koreicus │
│ anopheles          │
└────────────────────┘


Luego del sanity check, podemos confirmar que los datos fueron cargados exitosamente. Ahora, Yolo espera que las clases sean mappeadas de manera numérica.

In [6]:
classes = sorted(train_df.select("class_label").unique()["class_label"].to_list())
print("classes:", classes)

class_to_id = {cls: i for i, cls in enumerate(classes)}
print("class_to_id:", class_to_id)

train_df = train_df.with_columns(
    pl.col("class_label")
      .replace(class_to_id)      # <- use dict mapping instead of map_elements
      .cast(pl.Int64)
      .alias("class_id")
)

val_df = val_df.with_columns(
    pl.col("class_label")
      .replace(class_to_id)
      .cast(pl.Int64)
      .alias("class_id")
)

test_df = test_df.with_columns(
    pl.col("class_label")
      .replace(class_to_id)
      .cast(pl.Int64)
      .alias("class_id")
)

train_df.head()


classes: ['aegypti', 'albopictus', 'anopheles', 'culex', 'culiseta', 'japonicus/koreicus']
class_to_id: {'aegypti': 0, 'albopictus': 1, 'anopheles': 2, 'culex': 3, 'culiseta': 4, 'japonicus/koreicus': 5}


img_fName,img_w,img_h,bbx_xtl,bbx_ytl,bbx_xbr,bbx_ybr,class_label,class_id
str,i64,i64,i64,i64,i64,i64,str,i64
"""92715872-3287-4bff-aa61-704797…",2448,3264,1301,1546,1641,2096,"""albopictus""",1
"""82df4b68-0f45-4afe-9215-48488b…",768,1024,220,58,659,808,"""albopictus""",1
"""331ad30a-7564-4478-b863-7bc760…",3456,4608,1169,2364,1586,2826,"""albopictus""",1
"""46f34803-f754-457d-bdb7-e581d3…",1152,2560,198,798,954,1351,"""albopictus""",1
"""5792dd8b-e690-4c3a-b061-a375ab…",3072,4080,1104,1030,2458,2911,"""anopheles""",2


In [7]:
test_df.head()

img_fName,img_w,img_h,bbx_xtl,bbx_ytl,bbx_xbr,bbx_ybr,class_label,class_id
str,i64,i64,i64,i64,i64,i64,str,i64
"""b0f7cc74-2272-4756-a387-38bcaf…",3024,4032,900,1897,1950,2990,"""albopictus""",1
"""707723f2-5391-4b44-9255-199233…",1024,1820,576,754,801,1143,"""culiseta""",4
"""87aa3106-f8cb-4d5e-b636-ee9ea9…",1024,1365,316,646,588,901,"""culex""",3
"""fcd44271-4d8a-4727-80df-e85746…",828,1792,342,760,565,1068,"""culex""",3
"""1104015d-f0fa-4a65-b08d-0335d2…",3024,4032,869,1017,2019,2588,"""japonicus/koreicus""",5


Ahora podemos empezar a construir los directorios para el  modelo

### Construcción Formato YOLO
Yolo tiene un formato específico que se debe seguir para entrenar sus  modelos, por lo cual debemos crear nuevos directorios y re-organizar nuestros datos. Empezando por crear nuestros paths, al igual que algunas otras variables de utilidad para evitar ser desordenados

### Recorte de Imágenes

Ahora tenemos que recortar las imágenes, según el bounding box. 

In [8]:
from pathlib import Path
import cv2

# Directorio donde guardaremos el dataset recortado para clasificación
CLS_DATA_ROOT = Path("/kaggle/working/mosquito_cls")

# Asegurarnos de que exista la estructura esperada por YOLOv8-Classification
for split in ["train", "val", "test"]:
    for cls in classes:  # 'classes' ya lo definiste antes
        out_dir = CLS_DATA_ROOT / split / cls
        out_dir.mkdir(parents=True, exist_ok=True)

CLS_DATA_ROOT, list((CLS_DATA_ROOT / "train").iterdir())[:3]


(PosixPath('/kaggle/working/mosquito_cls'),
 [PosixPath('/kaggle/working/mosquito_cls/train/aegypti'),
  PosixPath('/kaggle/working/mosquito_cls/train/culex'),
  PosixPath('/kaggle/working/mosquito_cls/train/anopheles')])

Ahora, podemos definir una función de utilidad para llenar los directorios dónde tenemos los datos en el formato que espera YOLO

In [9]:
def crop_and_save_split(df: pl.DataFrame, split: str):
    """
    Recorta las imágenes según el bbox y las guarda en:
    CLS_DATA_ROOT / split / class_name / archivo.jpg
    """
    if split in ("train", "val"):
        img_base = Path(RAW_DATA_PATH) / "train_images"
    else:
        img_base = Path(RAW_DATA_PATH) / "train_images"

    n_rows = df.height
    print(f"Procesando split='{split}' con {n_rows} filas...")

    for i, row in enumerate(df.iter_rows(named=True), start=1):
        img_fname = row["img_fName"]
        class_id  = row["class_id"]
        class_name = classes[class_id]

        img_path = img_base / img_fname
        img = cv2.imread(str(img_path))

        if img is None:
            print(f"[WARN] No se pudo leer la imagen: {img_path}")
            continue

        h, w = img.shape[:2]

        # Coordenadas del bbox
        x1 = int(row["bbx_xtl"])
        y1 = int(row["bbx_ytl"])
        x2 = int(row["bbx_xbr"])
        y2 = int(row["bbx_ybr"])

        # Limitar a los bordes de la imagen por seguridad
        x1 = max(0, min(x1, w - 1))
        y1 = max(0, min(y1, h - 1))
        x2 = max(0, min(x2, w))
        y2 = max(0, min(y2, h))

        # Asegurar que haya área positiva
        if x2 <= x1 or y2 <= y1:
            print(f"[WARN] BBox inválido para {img_fname}: ({x1},{y1})-({x2},{y2})")
            continue

        crop = img[y1:y2, x1:x2]

        # Nombre del archivo de salida (mantenemos el nombre base y añadimos _crop)
        out_name = f"{Path(img_fname).stem}.jpeg"
        out_path = CLS_DATA_ROOT / split / class_name / out_name

        cv2.imwrite(str(out_path), crop)

        if i % 1000 == 0:
            print(f"  -> {i}/{n_rows} imágenes procesadas para split='{split}'")

    print(f"Terminado split='{split}' ")


In [10]:
crop_and_save_split(train_df, "train")

Procesando split='train' con 6396 filas...
  -> 1000/6396 imágenes procesadas para split='train'
  -> 2000/6396 imágenes procesadas para split='train'
  -> 3000/6396 imágenes procesadas para split='train'
  -> 4000/6396 imágenes procesadas para split='train'
  -> 5000/6396 imágenes procesadas para split='train'
  -> 6000/6396 imágenes procesadas para split='train'
Terminado split='train' 


In [11]:
crop_and_save_split(val_df,   "val")

Procesando split='val' con 800 filas...
Terminado split='val' 


In [12]:
#crop_and_save_split(test_df,  "test")

Podemos correr unos sanity checks sobre los conjuntos de prueba y validación

In [13]:
for split in ["train", "val"]:
    print(f"\nSplit: {split}")
    for cls in classes:
        n_imgs = len(list((CLS_DATA_ROOT / split / cls).glob("*.jpeg")))
        print(f"  {cls:18s}: {n_imgs}")



Split: train
  aegypti           : 28
  albopictus        : 2845
  anopheles         : 51
  culex             : 2827
  culiseta          : 390
  japonicus/koreicus: 255

Split: val
  aegypti           : 4
  albopictus        : 356
  anopheles         : 6
  culex             : 353
  culiseta          : 49
  japonicus/koreicus: 32


Ahora como se puede ver existe un desbalance entre clases, por lo que tenemos que ajustar esto antes de entrenar el modelo.

In [16]:
from pathlib import Path
import random
import shutil

CLS_DATA_ROOT = Path("/kaggle/working/mosquito_cls")

def oversample_split(split: str, target: int | None = None):
    """
    Duplica imágenes de las clases minoritarias en CLS_DATA_ROOT/split
    hasta que todas tengan 'target' imágenes.

    - split: 'train', 'val', etc. (normalmente SOLO 'train')
    - target: número objetivo de imágenes por clase.
              Si es None, usa la clase con mayor número de imágenes.
    """
    print(f"\n=== Oversampling split='{split}' ===")

    # 1) Conteos actuales
    counts = {}
    for cls in classes:
        cls_dir = CLS_DATA_ROOT / split / cls  # ojo: cls puede ser 'japonicus/koreicus'
        imgs = list(cls_dir.glob("*.jpeg")) + list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png"))
        counts[cls] = len(imgs)

    print("Conteos antes:")
    for cls, n in counts.items():
        print(f"  {cls:18s}: {n}")

    # 2) Elegir target
    if target is None:
        target = max(counts.values())
    print(f"\nTarget por clase: {target} imágenes")

    # 3) Duplicar imágenes donde haga falta
    for cls in classes:
        cls_dir = CLS_DATA_ROOT / split / cls
        imgs = list(cls_dir.glob("*.jpeg")) + list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png"))
        n = len(imgs)

        if n == 0:
            print(f"  [WARN] clase '{cls}' sin imágenes, se omite.")
            continue
        if n >= target:
            continue  # ya está al nivel deseado

        orig_imgs = imgs.copy()
        k = 0
        while len(imgs) < target:
            src = random.choice(orig_imgs)
            k += 1
            new_name = f"{src.stem}_dup{k}{src.suffix}"
            dst = cls_dir / new_name
            shutil.copy2(src, dst)
            imgs.append(dst)

        print(f"  Clase '{cls}': {n} -> {len(imgs)} imágenes")

    # 4) Conteos después
    print("\nConteos después:")
    for cls in classes:
        cls_dir = CLS_DATA_ROOT / split / cls
        imgs = list(cls_dir.glob("*.jpeg")) + list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png"))
        print(f"  {cls:18s}: {len(imgs)}")

In [17]:
# Ejecutamos oversampling SOLO en train
oversample_split("train")   


=== Oversampling split='train' ===
Conteos antes:
  aegypti           : 28
  albopictus        : 2845
  anopheles         : 51
  culex             : 2827
  culiseta          : 390
  japonicus/koreicus: 255

Target por clase: 2845 imágenes
  Clase 'aegypti': 28 -> 2845 imágenes
  Clase 'anopheles': 51 -> 2845 imágenes
  Clase 'culex': 2827 -> 2845 imágenes
  Clase 'culiseta': 390 -> 2845 imágenes
  Clase 'japonicus/koreicus': 255 -> 2845 imágenes

Conteos después:
  aegypti           : 2845
  albopictus        : 2845
  anopheles         : 2845
  culex             : 2845
  culiseta          : 2845
  japonicus/koreicus: 2845


Verficamos que haya surtido efecto

In [18]:
for split in ["train", "val"]:
    print(f"\nSplit: {split}")
    for cls in classes:
        n_imgs = len(list((CLS_DATA_ROOT / split / cls).glob("*.jpeg")))
        print(f"  {cls:18s}: {n_imgs}")


Split: train
  aegypti           : 2845
  albopictus        : 2845
  anopheles         : 2845
  culex             : 2845
  culiseta          : 2845
  japonicus/koreicus: 2845

Split: val
  aegypti           : 4
  albopictus        : 356
  anopheles         : 6
  culex             : 353
  culiseta          : 49
  japonicus/koreicus: 32


Podemos ver que ya tenemos todo cargado, por lo que podemos empezar a entrenar el modelo

## Entrenamiento

In [19]:
import cv2
from ultralytics.utils import patches

def simple_imread(path, flags=cv2.IMREAD_COLOR):
    return cv2.imread(str(path), flags)

patches.imread = simple_imread
print("Patched ultralytics imread -> cv2.imread")

Patched ultralytics imread -> cv2.imread


In [20]:
CLS_ROOT = Path("/kaggle/working/mosquito_cls")

print("CLS_ROOT contents:", os.listdir(CLS_ROOT))
print("train classes:", os.listdir(CLS_ROOT / "train"))

CLS_ROOT contents: ['val', 'test', 'train']
train classes: ['aegypti', 'culex', 'anopheles', 'culiseta', 'japonicus', 'albopictus']


In [22]:
from ultralytics import YOLO
import torch
from pathlib import Path

IMG_SIZE = 224
model = YOLO("yolov8s-cls.pt")   

results = model.train(
    data=str(CLS_ROOT),                   
    imgsz=IMG_SIZE,
    epochs=25,                              
    batch=64,                               
    workers=0,                              
    device=0 if torch.cuda.is_available() else "cpu",
    amp=False,                              
    patience=5,                             
    verbose=True,
    project=str(CLS_ROOT / "runs"),             
    name="yolov8n_mosquitos"               
)

Ultralytics 8.3.228 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/mosquito_cls, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_mosquitos, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, perspective=0.0, plots=

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all      0.923          1
Speed: 0.1ms preprocess, 0.6ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /kaggle/working/mosquito_cls/runs/yolov8n_mosquitos


Ahora, podemos guardar el modelo a los outputs de Kaggle

## Guardado del Modelo
Ahora, para poder guardar nuestro modelo podemos almacenar los mejores weights encontrados dentro del working directory de Kaggle. De esta manera, podemos descargarlos fácilmente y correr el modelo sin necesidad de volver a entrenarlo.

In [23]:
from pathlib import Path
import shutil

# Directorio donde YOLO guardó este experimento
save_dir = Path(results.save_dir)
print("Save dir:", save_dir)

# 1) Pesos
best_src = save_dir / "weights" / "best.pt"
last_src = save_dir / "weights" / "last.pt"

best_dst = Path("/kaggle/working/mosquito_cls_best.pt")
last_dst = Path("/kaggle/working/mosquito_cls_last.pt")

shutil.copy(best_src, best_dst)
shutil.copy(last_src, last_dst)

print("Saved best model to:", best_dst)
print("Saved last model to:", last_dst)

# 2) Artefactos adicionales
extra_files = [
    "results.png",          # curvas de entrenamiento
    "results.csv",          # métricas por época
    "confusion_matrix.png",
    "PR_curve.png",
    "labels.jpg",
]

for name in extra_files:
    src = save_dir / name
    if src.exists():
        dst = Path("/kaggle/working") / name
        shutil.copy(src, dst)
        print(f"Saved {name} to: {dst}")
    else:
        print(f"{name} not found in {save_dir}, skipping.")


Save dir: /kaggle/working/mosquito_cls/runs/yolov8n_mosquitos
Saved best model to: /kaggle/working/mosquito_cls_best.pt
Saved last model to: /kaggle/working/mosquito_cls_last.pt
Saved results.png to: /kaggle/working/results.png
Saved results.csv to: /kaggle/working/results.csv
Saved confusion_matrix.png to: /kaggle/working/confusion_matrix.png
PR_curve.png not found in /kaggle/working/mosquito_cls/runs/yolov8n_mosquitos, skipping.
labels.jpg not found in /kaggle/working/mosquito_cls/runs/yolov8n_mosquitos, skipping.
